**O3 模型・重點小結**

1. **核心問題**

   * 你要分析一批文本，想先用 **GAP**（Generalized Association Plots）做可視化分群，再用 **LDA**（Latent Dirichlet Allocation）或其監督式變體（如 Labeled LDA）來取得每篇文本的主題分佈，判斷「偏向哪個主題」。

2. **兩種流程比較**

   | 流程 | 先做 GAP → 再做 LDA         | 先指定 topic→token → GAP 分群 → LDA 驗證 |
   | -- | ----------------------- | --------------------------------- |
   | 用途 | 讓可視化主動揭示隱含群，再以 LDA 量化主題 | 用專家/先驗定義 token 群，再檢驗實際分布          |
   | 優點 | 完全資料驅動，能發現意外結構          | 可驗證人為標籤、易對照 domain 知識             |
   | 風險 | GAP 聚類不一定對應可解釋主題        | 預先 token list 可能偏誤，限制 LDA 彈性      |

3. **主題歸屬 → 延伸應用**

   * **推薦系統**：把「文件–主題分布」對應成「使用者/商品–潛在偏好」，即可做內容推薦或商品推薦。
   * **顧客分析**：將每位顧客的行為紀錄視為文件，LDA 得到「顧客–主題偏好向量」，GAP 幫你做群組視覺化，便於客群分層、行銷定位。

4. **實作建議**

   1. **訓練**：對已標或未標文本跑 LDA（或 Labeled LDA 如果有多重標籤）。
   2. **推論**：輸入新文本，推斷其 `θ`（topic mixture），即得到「偏向哪個主題」的機率。
   3. **可視化／分群**：用 GAP 依 `θ` 或相似度排序，觀察群落與主題塊狀。
   4. **應用**：

      * 推薦：以使用者對主題的偏好去匹配文件或商品。
      * 分群：以顧客在主題空間的位置做 segmentation。

5. **下一步**

   * 若想快速驗證，可先用現成 LDA 套件（Gensim／sklearn）＋ R 版 GAP。
   * 需要設計推薦或顧客分析流程，再進一步整合「主題 → 相似度 → 排薦」的演算法。


### 有還原字型

In [ ]:
# import pandas as pd
# import re, string
# import numpy as np
# np.random.seed(42)

# import spacy
# nlp = spacy.load("en_core_web_sm")
# stopwords = nlp.Defaults.stop_words

# data = pd.read_csv(r'C:\Users\User\Documents\GitHub\psychic-spoon\BERTopic\bbc-news-data.csv', sep='\t')

# def preprocess(text):
#     text = text.lower()  # 轉成小寫
#     text = re.sub(r'\d+', '', text)  # 移除數字
#     text = re.sub(r'[^\w\s]', '', text)  # 移除標點符號
#     text = re.sub(r'\s+', ' ', text).strip()  # 移除多餘空白
    
#     doc = nlp(text)
#     lemmatized_tokens = [token.lemma_ for token in doc if token.is_alpha and token.lemma_ not in stopwords and len(token.lemma_) > 1]
#     return lemmatized_tokens

# data['Tokens'] = data['content'].apply(preprocess)
# data['cleaned_content'] = data['Tokens'].apply(lambda x: ' '.join(x))
# data.to_csv('BBCnews_cleaned.csv', encoding='utf-8-sig')

### 沒有還原字型

In [ ]:
# import pandas as pd
# import re, string
# import numpy as np
# np.random.seed(42)
# # 下載並設定停用詞
# import nltk
# from nltk.corpus import stopwords
# nltk.download('stopwords')
# stop_words = stopwords.words('english')
# custom_stop = {'said', 'mr', 'also', 'would', 'new', 'one', 'like', 'could', 'us', 'many', 'get', 'use', 'two', 'first',
#                'told', 'say', 'may', 'good', 'last', 'make', 'well', 'next'}
# all_stopwords = set(stop_words).union(custom_stop)

# data = pd.read_csv(r'C:\Users\User\Documents\GitHub\psychic-spoon\BERTopic\bbc-news-data.csv', sep='\t')

# def preprocess(text):
#     text = text.lower()  # 轉成小寫
#     text = re.sub(r'\d+', '', text)  # 移除數字
#     text = re.sub(r'[^\w\s]', '', text)  # 移除標點符號
#     text = re.sub(r'\s+', ' ', text).strip()  # 移除多餘空白
#     text = ' '.join([word for word in text.split() if word not in all_stopwords])  # 移除停用詞
#     return text

# data['cleaned_content'] = data['content'].apply(preprocess)
# data['Tokens'] = data['cleaned_content'].apply(lambda x: x.split())

---

---

In [23]:
data = pd.read_csv(r'C:\Users\USER\Documents\GitHub\psychic-spoon\BERTopic\BBCnews_cleaned.csv')
# 你把 list 存成 .csv 之後，pandas.read_csv 會把它當成 字串，不會還原回原本的 list。
# 所以
import ast
data['Tokens'] = data['Tokens'].apply(ast.literal_eval)


In [ ]:
data['category'].value_counts()

---

---

### 算文章的tfidf來設定seed

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

for label in data['category'].unique():
    docs = data[data['category'] == label]['cleaned_content']
    vec = TfidfVectorizer(max_df=0.8, min_df=5, stop_words='english')
    tfidf = vec.fit_transform(docs)
    top_words = sorted(zip(vec.get_feature_names_out(), 
                           tfidf.sum(axis=0).A1),
                           key=lambda x: -x[1])[:30]
    print(f'\'{label}\': {[w for w, _ in top_words]}' +',')

In [14]:
seed = {
'business': ['bn', 'year', 'company', 'mr', 'firm', 'sale', 'rise', 'bank', 'market', 'share', 'price', 'growth', 'economy', 'rate', 'government', 'new', 'fall', 'oil', 'economic', 'profit', 'report', 'month', 'analyst', 'business', 'high', 'expect', 'dollar', 'figure', 'deal', 'world'],
'entertainment': ['film', 'good', 'award', 'year', 'star', 'music', 'win', 'band', 'include', 'number', 'new', 'album', 'festival', 'actor', 'play', 'mr', 'oscar', 'director', 'chart', 'tv', 'uk', 'bbc', 'record', 'song', 'million', 'time', 'release', 'british', 'people', 'singer'],
'politics': ['mr', 'labour', 'election', 'party', 'blair', 'government', 'tory', 'people', 'minister', 'lord', 'brown', 'plan', 'howard', 'tax', 'new', 'tell', 'year', 'chancellor', 'prime', 'public', 'britain', 'uk', 'home', 'mp', 'campaign', 'country', 'issue', 'vote', 'leader', 'work'],
'sport': ['win', 'play', 'game', 'england', 'player', 'good', 'year', 'world', 'club', 'match', 'time', 'team', 'come', 'cup', 'champion', 'second', 'injury', 'think', 'wale', 'ireland', 'open', 'season', 'want', 'coach', 'final', 'chelsea', 'rugby', 'set', 'new', 'france'],
'tech': ['game', 'people', 'mobile', 'phone', 'use', 'technology', 'service', 'mr', 'user', 'music', 'software', 'new', 'computer', 'year', 'site', 'digital', 'firm', 'network', 'net', 'broadband', 'microsoft', 'player', 'search', 'million', 'email', 'online', 'company', 'tv', 'program', 'pc'],
}


In [ ]:

selected_tokens = set()
for words in seed.values():
    selected_tokens.update(words)

from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer(vocabulary=list(selected_tokens))
X = vectorizer.fit_transform(data['cleaned_content'])

tf_df = pd.DataFrame(X.toarray(), columns=vectorizer.get_feature_names_out())
tf_df.to_csv("還原字型0.8_5的tfidf前30.txt", index=False, sep='\t')


---

---

---

In [15]:
from gensim.corpora import Dictionary
from gensim.models.ldamodel import LdaModel
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, adjusted_rand_score, classification_report
import pandas as pd
import numpy as np

# 1. 建立詞典與語料

dictionary = Dictionary(data['Tokens'])
corpus = [dictionary.doc2bow(tokens) for tokens in data['Tokens']]

# 2. 讀取 token-to-topic mapping
token_df = pd.read_csv('還原字型0.8_5的tfidf前30手動分群.csv')
token_to_topic = dict(zip(token_df['token'], token_df['topic']))

# 3. 建立 η 矩陣
K = 5
eta_strength=0.3

V = len(dictionary.token2id)
eta = np.full((K, V), 0.01)
for word, topic_id in token_to_topic.items():
    if topic_id < K and word in dictionary.token2id:
        word_id = dictionary.token2id[word]
        eta[topic_id][word_id] = eta_strength

# 4. 建立 LDA 模型
lda = LdaModel(corpus=corpus, id2word=dictionary, num_topics=K,
               eta=eta, random_state=42, iterations=100, passes=10, alpha=0.1)

# lda.save('20250719lda_model.model')
# dictionary.save('20250719dictionary.dict')

# 5. 主題詞顯示
print("\n每個主題的前 20 詞：")
for k in range(K):
    top_words = lda.show_topic(k, topn=20)
    print(f"Topic {k+1}: {[w for w, _ in top_words]}")

covered_words = [word for word in token_to_topic if word in dictionary.token2id]
print(f"你的 token_to_topic 裡共 {len(token_to_topic)} 個詞，其中 {len(covered_words)} 個有成功對應到 dictionary。")



每個主題的前 20 詞：
Topic 1: ['game', 'use', 'people', 'computer', 'technology', 'pc', 'user', 'search', 'software', 'new', 'mr', 'time', 'machine', 'microsoft', 'find', 'work', 'year', 'firm', 'play', 'release']
Topic 2: ['year', 'film', 'good', 'sale', 'win', 'award', 'pp', 'include', 'new', 'music', 'play', 'bn', 'star', 'number', 'rise', 'uk', 'price', 'title', 'release', 'time']
Topic 3: ['mr', 'year', 'government', 'world', 'people', 'time', 'tell', 'new', 'game', 'play', 'come', 'win', 'think', 'plan', 'minister', 'labour', 'work', 'need', 'party', 'election']
Topic 4: ['phone', 'mobile', 'people', 'service', 'technology', 'digital', 'music', 'use', 'tv', 'new', 'broadband', 'consumer', 'player', 'year', 'video', 'million', 'device', 'content', 'market', 'gadget']
Topic 5: ['firm', 'site', 'company', 'attack', 'system', 'email', 'use', 'people', 'security', 'net', 'software', 'legal', 'network', 'file', 'mr', 'website', 'spam', 'virus', 'user', 'law']
你的 token_to_topic 裡共 123 個詞，其中 12

In [17]:
import joblib

# 打包所有重要物件成一個 workspace dict
workspace = {
    'df': data,  # 你的 DataFrame
    'tokens': data['Tokens'],  # 你的 Tokens
    'dictionary': dictionary,  # gensim Dictionary
    'corpus': corpus,  # gensim Corpus
    'lda_model': lda # gensim LDA Model
}

# 儲存整個 workspace
joblib.dump(workspace, '20250719_0.8_5_30.pkl')


['20250719_0.8_5_30.pkl']

In [ ]:
from gensim.models import LdaModel
from gensim.corpora import Dictionary

lda = LdaModel.load('20250719lda_model.model')
dictionary = Dictionary.load('20250719dictionary.dict')


In [ ]:
import pandas as pd

doc_topics = lda.get_document_topics(corpus, minimum_probability=0)

topic_distributions = pd.DataFrame([
    [prob for _, prob in doc] for doc in doc_topics
])

topic_distributions.columns = [f"Topic_{i+1}" for i in range(K)]
topic_distributions['BBC_Label'] = data['category'].values

from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
import seaborn as sns

tsne = TSNE(n_components=2, random_state=42)
X_embedded = tsne.fit_transform(topic_distributions.iloc[:, :K].values)

topic_distributions['x'] = X_embedded[:, 0]
topic_distributions['y'] = X_embedded[:, 1]

plt.figure(figsize=(8,6))
sns.scatterplot(data=topic_distributions, x='x', y='y', hue='BBC_Label', palette='tab10')
plt.title('LDA Topic Distribution (t-SNE Projection) by BBC Category')
plt.show()


In [ ]:
# 拿每篇文件的 topic 分布（含所有 topic，minimum_probability=0 確保每個都有出現）
doc_topics = lda.get_document_topics(corpus, minimum_probability=0)

topic_distributions = pd.DataFrame([[prob for _, prob in doc] for doc in doc_topics])
topic_distributions.columns = [f"Topic_{i+1}" for i in range(K)]  # 從 1 開始好看一點
topic_distributions['BBC_Label'] = data['category'].values  # 加上原 BBC 分類方便比對

print(topic_distributions.head())


In [ ]:
mean_distribution_per_label = topic_distributions.groupby('BBC_Label').mean()
print(mean_distribution_per_label.T) 

In [ ]:
topic_distributions

In [ ]:
from scipy.spatial.distance import cdist
from sklearn.metrics import accuracy_score, classification_report, adjusted_rand_score

# 1️⃣ 準備 one-hot topic vectors
one_hot_topics = np.eye(K)  # 產生 K x K 單位矩陣，每一列就是一個 one-hot vector

# 2️⃣ 將原 dataframe 的 LDA topic 機率轉為 numpy array
lda_probs = topic_distributions.iloc[:, :K].values  # 取前 K 欄 (Topic_1, Topic_2...)

# 3️⃣ 計算每個文件對每個 one-hot 的距離
distances = cdist(lda_probs, one_hot_topics, metric='euclidean')  # (N x K)

# 4️⃣ 找出最小距離的 one-hot index，視為該文件屬於該 topic
pred_topic_by_distance = np.argmin(distances, axis=1)+1 # 1-based topic index

# 5️⃣ BBC label 轉成數值 (0 ~ 4)，方便比較
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
true_label = le.fit_transform(data['category'])

# # 6️⃣ 評估效果
# acc = accuracy_score(true_label, pred_topic_by_distance)
# ari = adjusted_rand_score(true_label, pred_topic_by_distance)
# report = classification_report(true_label, pred_topic_by_distance, target_names=le.classes_)

# print("\nAccuracy:", round(acc, 4))
# print("Adjusted Rand Index (ARI):", round(ari, 4))
# print("\nClassification Report:\n", report)


In [ ]:
pred_topic_by_distance

In [ ]:
import numpy as np
import pandas as pd

mapped_df = pd.DataFrame({
    'true_label': data['category'].values,
    'pred_topic': pred_topic_by_distance
})

print(mapped_df.groupby(['true_label', 'pred_topic']).size())


In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics.pairwise import cosine_similarity
from scipy.spatial.distance import cdist

# 你的 LDA topic 分布（N x K matrix）
lda_probs = topic_distributions.iloc[:, :K].values

# ground truth label to one-hot (vec-y)
N = len(true_label)
K = len(le.classes_)

Y_onehot = np.zeros((N, K))
Y_onehot[np.arange(N), true_label] = 1


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

consistency_scores = []

for y_true, y_hat in zip(Y_onehot, lda_probs):
    score = cosine_similarity([y_true], [y_hat])[0][0]
    consistency_scores.append(score)

print("Average Consistency (y vs y_hat):", round(np.mean(consistency_scores), 4))


In [ ]:
category_centroids = {}

for idx, label in enumerate(le.classes_):
    mask = (true_label == idx)
    centroid = lda_probs[mask].mean(axis=0)
    category_centroids[label] = centroid

centroid_matrix = np.vstack(list(category_centroids.values()))

# 類別中心兩兩 cosine 相似度
similarity_matrix = cosine_similarity(centroid_matrix)

# 類別中心兩兩 L2 距離
distance_matrix = cdist(centroid_matrix, centroid_matrix, metric='euclidean')

centroid_df = pd.DataFrame(similarity_matrix, index=le.classes_, columns=le.classes_)
print("\nCategory Center Cosine Similarity:")
print(centroid_df)

distance_df = pd.DataFrame(distance_matrix, index=le.classes_, columns=le.classes_)
print("\nCategory Center L2 Distance:")
print(distance_df)


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

sns.heatmap(centroid_df, annot=True, cmap='coolwarm')
plt.title("Cosine Similarity Between Category Topic Centers")
plt.show()

sns.heatmap(distance_df, annot=True, cmap='Blues')
plt.title("L2 Distance Between Category Topic Centers")
plt.show()
